In [1]:
# 1. Code to extract all the funds data.
# Note: Please run for 4-5 funds and save them in a csv file, then again change the runtime to GPU or TPU and do the same again.
# Please have 2 files that were extracted from this notebook. We will merge in the main notebook file.

!pip install mstarpy
import numpy as np
import datetime
import pandas as pd
import mstarpy
import time
from tqdm import tqdm

# Define list of funds to analyze
funds = [
    # "VBIAX",   # Vanguard Balanced Index Fund Admiral Shares (Vanguard)
    # "FBALX",   # Fidelity Balanced Fund (Fidelity)
    # "ABALX",   # American Funds American Balanced Fund (American Funds)
    # "PRWCX",   # T. Rowe Price Capital Appreciation Fund (T. Rowe Price)
    "BAICX",   # BlackRock Global Allocation Fund (BlackRock)
    "OAKBX",   # Oakmark Equity and Income Fund (Oakmark)
    "DODBX",   # Dodge & Cox Balanced Fund (Dodge & Cox)
    "JABAX",   # Janus Henderson Balanced Fund (Janus Henderson)
    "GLRBX",   # Goldman Sachs Balanced Strategy Portfolio (Goldman Sachs)
    "TRRIX"    # TIAA-CREF Lifecycle Index Retire Income Fund (TIAA)
]

# Define date range for the analysis
start_date = datetime.datetime(2003, 1, 1)
end_date = datetime.datetime(2025, 12, 31)

# Risk-free rate (adjust as necessary)
risk_free_rate = 0.00

# Function to calculate MDD manually
def calculate_mdd_for_year(group):
    """
    Calculate the Maximum Drawdown (MDD) for a given year's returns manually.
    """
    # Convert daily returns to cumulative returns
    cumulative_returns = (group['daily_return'] + 1).cumprod()
    # Avoid extreme or invalid cumulative return values
    cumulative_returns = cumulative_returns.replace([np.inf, -np.inf], np.nan).fillna(1)
    # Calculate rolling maximum of cumulative returns
    rolling_max = cumulative_returns.cummax()
    # Calculate drawdowns
    drawdowns = (cumulative_returns - rolling_max) / rolling_max
    # Find the maximum drawdown
    mdd = drawdowns.min()
    return mdd

# Function to calculate metrics for each year
def calculate_yearly_metrics(group):
    # Calculate yearly return
    yearly_return = (group['daily_return'] + 1).prod() - 1
    # Avoid invalid yearly returns
    if np.isinf(yearly_return) or np.isnan(yearly_return):
        yearly_return = -1
    # Calculate volatility (standard deviation of daily returns)
    volatility = group['daily_return'].std() * np.sqrt(252)  # Annualized volatility
    # Calculate Sharpe Ratio
    sharpe_ratio = (yearly_return - risk_free_rate) / volatility if volatility != 0 else np.nan
    # Calculate MDD
    mdd = calculate_mdd_for_year(group)
    return pd.Series({
        'Annual_Return': yearly_return,
        'Volatility': volatility,
        'Sharpe_Ratio': sharpe_ratio,
        'MDD': mdd
    })

# Initialize master dataframe to store results
master_df = pd.DataFrame()

# Process each fund
print("Fetching and processing fund data...")
for fund_ticker in tqdm(funds):
    try:
        print(f"\nProcessing {fund_ticker}")

        # Initialize fund object
        fund = mstarpy.Funds(term=fund_ticker, country="us")

        # Get historical data
        print(f"Fetching historical data for {fund_ticker}...")
        history = fund.nav(start_date=start_date, end_date=end_date, frequency="daily")

        # Convert to pandas DataFrame
        df = pd.DataFrame(history)

        # Skip to next fund if data retrieval failed
        if df.empty:
            print(f"No data retrieved for {fund_ticker}. Skipping to next fund.")
            time.sleep(3)  # Delay before next fund
            continue

        # Ensure date column is in datetime format and sort by date
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values(by='date')

        # Calculate daily returns
        df['daily_return'] = df['nav'].pct_change()

        # Drop NaN values created by pct_change()
        df = df.dropna(subset=['daily_return'])

        # Extract year from date
        df['Year'] = df['date'].dt.year

        # Fill NaN values
        df = df.fillna(0)

        # Calculate yearly metrics
        yearly_metrics = df.groupby('Year').apply(calculate_yearly_metrics).reset_index()

        # Add fund ticker to the metrics
        yearly_metrics['Fund'] = fund_ticker

        # Append to master dataframe
        master_df = pd.concat([master_df, yearly_metrics[['Year', 'Fund', 'Annual_Return', 'Sharpe_Ratio']]])

        print(f"Successfully processed {fund_ticker}")

        # Delay before processing next fund to avoid API rate limits
        time.sleep(5)

    except Exception as e:
        print(f"Error processing {fund_ticker}: {str(e)}")
        time.sleep(3)  # Shorter delay if there was an error
        continue

# Convert Year to integer
master_df['Year'] = master_df['Year'].astype(int)

# Calculate ranks for each year
print("Calculating performance rankings...")
final_df = pd.DataFrame()

for year in master_df['Year'].unique():
    year_data = master_df[master_df['Year'] == year].copy()

    # Calculate ranks (1 is best)
    year_data['Return_Rank'] = year_data['Annual_Return'].rank(ascending=False).astype(int)
    year_data['Sharpe_Rank'] = year_data['Sharpe_Ratio'].rank(ascending=False).astype(int)

    # Append to final dataframe
    final_df = pd.concat([final_df, year_data])

# Format percentage columns
final_df['Annual_Return'] = final_df['Annual_Return'].apply(lambda x: f"{x:.2%}")
final_df['Sharpe_Ratio'] = final_df['Sharpe_Ratio'].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")

# Reorder columns
final_df = final_df[['Year', 'Fund', 'Annual_Return', 'Sharpe_Ratio', 'Return_Rank', 'Sharpe_Rank']]

# Sort by Year (ascending) and Fund
final_df = final_df.sort_values(by=['Year', 'Fund'])

# Display final table
print("\nFinal Performance Table:")
print(final_df)

# Save results to CSV
final_df.to_csv("mutual_fund_performance.csv", index=False)
print("\nResults saved to 'mutual_fund_performance_1.csv'")

Fetching and processing fund data...


  0%|          | 0/6 [00:00<?, ?it/s]


Processing BAICX
Fetching historical data for BAICX...


<ipython-input-1-75c9bbcd0945>:109: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  yearly_metrics = df.groupby('Year').apply(calculate_yearly_metrics).reset_index()


Successfully processed BAICX


...


Processing TRRIX
Fetching historical data for TRRIX...


<ipython-input-1-75c9bbcd0945>:109: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  yearly_metrics = df.groupby('Year').apply(calculate_yearly_metrics).reset_index()


Successfully processed TRRIX


100%|██████████| 6/6 [00:37<00:00,  6.22s/it]

Calculating performance rankings...

Final Performance Table:
    Year   Fund Annual_Return Sharpe_Ratio  Return_Rank  Sharpe_Rank
0   2003  DODBX        20.23%         1.88            2            4
0   2003  GLRBX        16.08%         2.52            3            2
0   2003  JABAX        11.52%         1.48            5            5
0   2003  OAKBX        22.40%         2.58            1            1
0   2003  TRRIX        12.93%         2.12            4            3
..   ...    ...           ...          ...          ...          ...
22  2025  DODBX         4.26%         0.33            1            1
22  2025  GLRBX         0.90%         0.08            6            5
22  2025  JABAX         3.20%         0.19            2            4
22  2025  OAKBX         1.11%         0.07            5            6
22  2025  TRRIX         2.72%         0.31            3            2

[133 rows x 6 columns]

Results saved to 'mutual_fund_performance_1.csv'
